# Boolean Information Retrieval System
## Vocabulary Processing, Boolean Queries, and Tolerant Retrieval

**Course assignment notebook | Group 35 | BBC News full-text corpus**

This notebook is the executable companion to `docs/report/technical_report.md`. It follows the assignment pipeline exactly:

> **Remote corpus -> validation -> documents -> preprocessing -> vocabulary/dictionary -> inverted index -> sorted postings -> Boolean retrieval -> tolerant retrieval -> evaluation**

The retrieval logic is **not duplicated here**. Every experiment calls the tested implementation in `src/ir/`, so notebook execution and the current project process cannot drift apart. No RAG/LLM retrieval system or pre-built search engine is used.

## 1. Configure the remote file and processing parameters

The official BBC archive is the remote input. `DOWNLOAD_IF_MISSING=True` downloads only when `data/raw/bbc` is absent, which keeps submitted/offline Virtual Lab runs deterministic. Set `FORCE_DOWNLOAD=True` only to demonstrate a fresh remote retrieval.

All configurable paths and network settings are kept in the next cell.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    """Find the project root whether run from VS Code, Jupyter, or Virtual Lab."""
    candidates = [start.resolve(), *start.resolve().parents]
    for candidate in candidates:
        if (candidate / "src" / "ir").is_dir() and (candidate / "docs" / "IR_Assignment.md").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the IR-Assignment 1 repository.")


ROOT = find_repo_root(Path.cwd())
REMOTE_URL = "http://mlg.ucd.ie/files/datasets/bbc-fulltext.zip"
ARCHIVE_PATH = ROOT / "data" / "raw" / "bbc-fulltext.zip"
CORPUS_DIR = ROOT / "data" / "raw" / "bbc"
OUTPUT_DIR = ROOT / "data" / "derived" / "notebook"
REQUEST_TIMEOUT_SECONDS = 60
DOWNLOAD_CHUNK_BYTES = 1024 * 1024
DOWNLOAD_IF_MISSING = True
FORCE_DOWNLOAD = False
EXPECTED_ARCHIVE_SUFFIX = ".zip"
EXPECTED_DOCUMENT_COUNT = 2225

sys.path.insert(0, str(ROOT / "src")) if str(ROOT / "src") not in sys.path else None
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root : {ROOT}")
print(f"Remote source   : {REMOTE_URL}")
print(f"Corpus location : {CORPUS_DIR}")
print(f"Output location : {OUTPUT_DIR}")

## 2. Import required libraries

Only the dependencies already declared in `requirements.txt` are used. The download and archive steps use Python's standard library. NLTK data is loaded from the repository's bundled `nltk_data/` directory, so execution does not trigger an NLTK network download.

In [ ]:
import csv
import json
import os
import statistics
import subprocess
import time
import urllib.request
import zipfile
from collections import defaultdict

import matplotlib.pyplot as plt
import nltk

nltk_data_path = str(ROOT / "nltk_data")
if nltk_data_path not in nltk.data.path:
    nltk.data.path.insert(0, nltk_data_path)

from ir import boolean, evaluate, morphology, preprocess, queryset, tolerant
from ir.corpus import DATASET, corpus_stats, load_corpus
from ir.index import InvertedIndex
from ir.porter import trace


def show_table(rows, columns=None, limit=None):
    """Print compact records without adding a dataframe dependency."""
    rows = list(rows)
    if not rows:
        print("No rows.")
        return
    columns = columns or list(rows[0])
    shown = rows[:limit] if limit else rows
    rendered = [[str(row.get(column, "")) for column in columns] for row in shown]
    widths = [
        min(48, max(len(str(column)), *(len(values[position]) for values in rendered)))
        for position, column in enumerate(columns)
    ]

    def format_row(values):
        cells = []
        for value, width in zip(values, widths):
            shortened = value if len(value) <= width else value[: max(width - 3, 0)] + "..."
            cells.append(shortened.ljust(width))
        return " | ".join(cells)

    print(format_row([str(column) for column in columns]))
    print("-+-".join("-" * width for width in widths))
    for values in rendered:
        print(format_row(values))
    if len(shown) < len(rows):
        print(f"Showing {len(shown)} of {len(rows)} rows.")


print(f"Python {sys.version.split()[0]} | NLTK {nltk.__version__} | Matplotlib available")

## 3. Download the remote file

The download is streamed in fixed-size chunks and first written to a temporary `.part` file. `os.replace` publishes it atomically only after the HTTP response completes. The submitted repository already contains the corpus, so the normal notebook run reports a cache hit and remains offline.

In [ ]:
def download_archive(url: str, destination: Path) -> dict:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    request = urllib.request.Request(url, headers={"User-Agent": "IR-Assignment-Notebook/1.0"})
    try:
        with urllib.request.urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
            status = getattr(response, "status", 200)
            if status != 200:
                raise RuntimeError(f"Download failed with HTTP status {status}")
            content_type = response.headers.get_content_type()
            with temporary.open("wb") as output:
                while chunk := response.read(DOWNLOAD_CHUNK_BYTES):
                    output.write(chunk)
        os.replace(temporary, destination)
        return {"downloaded": True, "status": status, "content_type": content_type}
    except Exception:
        temporary.unlink(missing_ok=True)
        raise


should_download = FORCE_DOWNLOAD or (DOWNLOAD_IF_MISSING and not CORPUS_DIR.is_dir())
if should_download:
    download_info = download_archive(REMOTE_URL, ARCHIVE_PATH)
    print(f"Downloaded {ARCHIVE_PATH.stat().st_size:,} bytes to {ARCHIVE_PATH}")
else:
    download_info = {"downloaded": False, "status": "cached", "content_type": "local directory"}
    print("Using the existing local corpus; no network request was required.")

download_info

## 4. Validate and extract the downloaded file

A fresh archive must be non-empty, have the expected `.zip` suffix, and pass `zipfile.is_zipfile`. Extraction rejects archive entries that would escape `data/raw/`. The extracted corpus must contain exactly 2,225 article files, matching the documented BBC collection.

In [ ]:
def safe_extract_zip(archive: Path, destination: Path) -> None:
    destination_root = destination.resolve()
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            member_path = (destination / member.filename).resolve()
            if destination_root != member_path and destination_root not in member_path.parents:
                raise ValueError(f"Unsafe archive member: {member.filename}")
        bundle.extractall(destination)


if download_info["downloaded"]:
    assert ARCHIVE_PATH.exists(), "The downloaded archive does not exist."
    assert ARCHIVE_PATH.stat().st_size > 0, "The downloaded archive is empty."
    assert ARCHIVE_PATH.suffix.lower() == EXPECTED_ARCHIVE_SUFFIX, "Unexpected archive extension."
    assert zipfile.is_zipfile(ARCHIVE_PATH), "The downloaded file is not a valid ZIP archive."
    safe_extract_zip(ARCHIVE_PATH, ARCHIVE_PATH.parent)

article_paths = [
    path for path in CORPUS_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() == ".txt" and path.name.upper() != "README.TXT"
]
assert CORPUS_DIR.is_dir(), f"Corpus directory not found: {CORPUS_DIR}"
assert len(article_paths) == EXPECTED_DOCUMENT_COUNT, (
    f"Expected {EXPECTED_DOCUMENT_COUNT} BBC articles, found {len(article_paths)}"
)
print(f"Validation passed: {len(article_paths):,} BBC article files are ready.")

## 5. Load and profile the input data

`ir.corpus.load_corpus` reads UTF-8 article files in deterministic `(category, filename)` order and assigns dense, stable document IDs. It deliberately performs no text processing. The profile below is therefore the required before-processing baseline.

In [ ]:
docs = load_corpus(CORPUS_DIR)
stats = corpus_stats(docs)

assert len(docs) == EXPECTED_DOCUMENT_COUNT
assert [document.doc_id for document in docs] == list(range(len(docs)))
assert all(document.text for document in docs), "Every loaded document must contain text."

show_table([
    {"property": "Name", "value": DATASET["name"]},
    {"property": "Domain", "value": DATASET["domain"]},
    {"property": "Source", "value": DATASET["source"]},
    {"property": "Attribution", "value": DATASET["attribution"]},
    {"property": "Licence", "value": DATASET["licence"]},
    {"property": "Documents", "value": f"{stats['num_docs']:,}"},
    {"property": "Corpus size", "value": f"{stats['raw_bytes'] / 1e6:.1f} MB"},
    {"property": "Whitespace tokens", "value": f"{stats['total_whitespace_tokens']:,}"},
    {"property": "Unique whitespace types", "value": f"{stats['unique_whitespace_types']:,}"},
])

print("Document length:", stats["doc_length"])
print("First document:", docs[0].doc_id, docs[0].category, docs[0].filename)
print(docs[0].text[:240].replace("\n", " ") + "...")

## 6. Apply the assignment processing workflow
### 6.1 Tokenization, normalization, stop-word removal, stemming, and lemmatization

The five named configurations are cumulative. Tokenization uses the project's regular expression; Porter stemming is implemented locally; the stop-word list, POS tagger, and WordNet lemmatizer come from NLTK. POS tagging occurs before stop-word removal because the tagger needs sentence context.

In [ ]:
sample_text = "The Investors were ANNOUNCING new investments in US technologies."
preprocessing_rows = []
for configuration in preprocess.CONFIGURATIONS:
    tokens = configuration(sample_text)
    preprocessing_rows.append({
        "configuration": configuration.name,
        "token_count": len(tokens),
        "tokens": " ".join(tokens),
    })

show_table(preprocessing_rows)
assert [row["configuration"] for row in preprocessing_rows] == [
    "raw", "normalized", "nostop", "stemmed", "lemmatized"
]
assert preprocessing_rows[3]["tokens"] == "investor announc new invest us technologi"

### 6.2 Porter's stemming algorithm

`src/ir/porter.py` implements the complete 1980 sequence (1a, 1b, 1c, 2, 3, 4, 5a, 5b). The trace exposes each intermediate form and demonstrates that stemming may produce non-words; its purpose is term conflation, not linguistic correctness.

In [ ]:
trace_words = [
    "relational", "conditional", "investments", "managing", "technologies",
    "announced", "winning", "universities", "agreed", "feed",
]
porter_rows = [
    {"word": word, "trace": " -> ".join(f"{step}:{value}" for step, value in trace(word))}
    for word in trace_words
]
show_table(porter_rows)
assert dict(trace("relational"))["5b"] == "relat"
assert dict(trace("technologies"))["5b"] == "technologi"

### 6.3 Vocabulary, dictionary, inverted index, and sorted postings

For each configuration, `InvertedIndex.build` creates a separate sorted vocabulary with document frequency (`df`) and collection frequency (`cf`), plus strictly ascending, duplicate-free postings lists. `check_invariants()` verifies ordering, uniqueness, document-ID bounds, and agreement with recorded `df`.

In [ ]:
index_build_started = time.perf_counter()
indexes = {}
index_rows = []

for configuration in preprocess.CONFIGURATIONS:
    index = InvertedIndex.build(docs, configuration)
    index.check_invariants()
    indexes[configuration.name] = index
    index_rows.append(index.stats())

print(f"Built and validated five indexes in {time.perf_counter() - index_build_started:.1f} seconds.")
show_table(
    index_rows,
    ["index", "num_docs", "vocabulary_size", "total_postings", "total_tokens", "hapax_legomena", "mean_df"],
)

assert all(index.num_docs == len(docs) for index in indexes.values())
assert len(indexes["stemmed"].terms) < len(indexes["nostop"].terms)
assert len(indexes["lemmatized"].terms) < len(indexes["nostop"].terms)

**Interpretation.** Case normalization merges case variants without deleting tokens. Stop-word removal removes very frequent terms, reducing tokens and postings much more than vocabulary. Porter stemming is the most aggressive conflator and therefore produces the smallest vocabulary; WordNet lemmatization is more conservative.

In [ ]:
stemmed_index = indexes["stemmed"]
surface_index = indexes["nostop"]
stemmed_normalize = boolean.normalizer_for(preprocess.STEMMED)

sample_terms = ["oil", "elect", "broadband", "chelsea", "technolog"]
for term in sample_terms:
    print(stemmed_index.sample_postings(term))

for term in sample_terms:
    postings = stemmed_index.postings(term)
    assert postings == sorted(set(postings)), f"Invalid postings list for {term}"

### 6.4 Boolean retrieval

The recursive-descent parser enforces precedence `NOT > AND > OR`; parentheses override it, and adjacent terms imply `AND`. Every Boolean operation uses two-pointer merges over the sorted postings lists created above. Malformed queries raise `QuerySyntaxError`.

In [ ]:
boolean_queries = [
    "oil AND price",
    "football OR rugby",
    "bank AND NOT football",
    "(microsoft OR apple) AND software",
    "market prices oil gas",  # implicit AND
]
boolean_rows = []
for query_text in boolean_queries:
    expression = boolean.parse(query_text)
    documents, execution = boolean.execute(
        query_text, stemmed_index, stemmed_normalize, optimize=False
    )
    boolean_rows.append({
        "query": query_text,
        "parsed_expression": repr(expression),
        "hits": len(documents),
        "comparisons": execution.comparisons,
        "sample_doc_ids": documents[:8],
    })

show_table(boolean_rows)
assert set(boolean.execute("oil price", stemmed_index, stemmed_normalize)[0]) == set(
    boolean.execute("oil AND price", stemmed_index, stemmed_normalize)[0]
)

### 6.5 Normal versus postings-list-length-based processing

The naive strategy evaluates `AND` operands left to right. The optimized strategy orders positive operands by estimated postings length and defers `NOT`, allowing `difference(a, b)` to avoid materializing the full complement. Comparison counts are exact. Timings use the report's protocol: two warm-ups, nine measured runs, and alternating execution order to reduce cache bias.

In [ ]:
TIMING_REPEATS = 9


def timed_pair(query_text, index, normalize):
    boolean.execute(query_text, index, normalize, optimize=False)
    boolean.execute(query_text, index, normalize, optimize=True)
    naive_times, optimized_times = [], []
    for repeat in range(TIMING_REPEATS):
        if repeat % 2 == 0:
            naive_docs, naive_stats = boolean.execute(query_text, index, normalize, optimize=False)
            optimized_docs, optimized_stats = boolean.execute(query_text, index, normalize, optimize=True)
        else:
            optimized_docs, optimized_stats = boolean.execute(query_text, index, normalize, optimize=True)
            naive_docs, naive_stats = boolean.execute(query_text, index, normalize, optimize=False)
        naive_times.append(naive_stats.elapsed_ms)
        optimized_times.append(optimized_stats.elapsed_ms)
    assert set(naive_docs) == set(optimized_docs), "Execution strategies disagree."
    return {
        "query": query_text,
        "hits": len(naive_docs),
        "naive_comparisons": naive_stats.comparisons,
        "optimized_comparisons": optimized_stats.comparisons,
        "saving_pct": round(100 * (naive_stats.comparisons - optimized_stats.comparisons) / max(naive_stats.comparisons, 1), 1),
        "naive_ms": round(statistics.median(naive_times), 4),
        "optimized_ms": round(statistics.median(optimized_times), 4),
        "naive_order": " ".join(naive_stats.order),
        "optimized_order": " ".join(optimized_stats.order),
    }


optimization_queries = [
    "oil AND price",
    "bank AND NOT football",
    "(microsoft OR apple) AND software",
    "market AND prices AND oil AND gas",
]
optimization_rows = [
    timed_pair(query_text, stemmed_index, stemmed_normalize)
    for query_text in optimization_queries
]
show_table(optimization_rows)
assert all(row["naive_comparisons"] >= row["optimized_comparisons"] for row in optimization_rows)
assert next(row for row in optimization_rows if "NOT" in row["query"])["saving_pct"] > 80

### 6.6 Stemming versus lemmatization

Stemming aggressively removes suffixes and can improve morphological recall, but it may merge unrelated words (over-stemming) or miss irregular forms (under-stemming). WordNet lemmatization maps toward real dictionary forms and handles some irregulars, but is more conservative. The following probes run the real project implementations; the verdicts are computed, not hand-written.

In [ ]:
morphology_rows = morphology.probe_rows()
undesirable_rows = morphology.undesirable(morphology_rows)
show_table(
    undesirable_rows,
    ["word_a", "word_b", "should_merge", "stem_a", "stem_b", "lemma_a", "lemma_b", "verdict"],
)
print(f"Undesirable outcomes: {len(undesirable_rows)} of {len(morphology_rows)} probes")
assert any(row["word_a"] == "communism" and row["same_stem"] == "yes" for row in undesirable_rows)
assert any(row["word_a"] == "children" and row["same_lemma"] == "yes" for row in undesirable_rows)

In [ ]:
stem_groups = defaultdict(set)
lemma_groups = defaultdict(set)
for term in surface_index.terms:
    stem_groups[preprocess._cached_stem(term)].add(term)
    lemma_groups[preprocess.context_free_lemma(term)].add(term)

conflation_rows = []
for stem, members in sorted(stem_groups.items(), key=lambda item: (-len(item[1]), item[0])):
    if len(members) < 2:
        continue
    lemmas = {preprocess.context_free_lemma(member) for member in members}
    conflation_rows.append({
        "stem": stem,
        "surface_forms": len(members),
        "distinct_lemmas": len(lemmas),
        "conflation_ratio": round(len(members) / len(lemmas), 2),
        "members": " ".join(sorted(members)[:12]),
    })

show_table(conflation_rows, limit=12)
print(
    f"Stem groups: {len(stem_groups):,} (mean {statistics.fmean(map(len, stem_groups.values())):.2f} forms) | "
    f"Lemma groups: {len(lemma_groups):,} (mean {statistics.fmean(map(len, lemma_groups.values())):.2f} forms)"
)

### 6.7 Tolerant retrieval: 3-grams, wildcards, and edit distance

One 3-gram index is built over the **un-stemmed `nostop` vocabulary**. Wildcards first intersect k-gram candidate sets, then apply a regex for exact pattern order. Misspellings are shortlisted by shared k-grams and ranked by capped Levenshtein distance, with collection frequency breaking ties.

Correction must run before stemming: stemming `goverment` first yields the existing stem `gover`, causing a stem-vocabulary corrector to miss `government`.

In [ ]:
kgram = tolerant.KGramIndex(surface_index.terms, k=3)
kgram.attach_frequencies(surface_index.vocabulary.collection_frequency)

wildcard_rows = []
for pattern in ["re*val", "comput*", "*ology", "invest*", "gov*ment", "f*ball"]:
    matches = kgram.wildcard(pattern)
    wildcard_rows.append({
        "pattern": pattern,
        "matching_terms": len(matches),
        "sample": " ".join(matches[:12]),
    })
show_table(wildcard_rows)

wildcard_query_rows = []
for query_text in [
    "comput* AND security",
    "invest* AND NOT football",
    "(broadband OR mobile) AND *phone",
    "elect* AND labour",
]:
    hits, execution = boolean.execute(
        query_text,
        stemmed_index,
        stemmed_normalize,
        optimize=True,
        expand=kgram.wildcard,
    )
    wildcard_query_rows.append({
        "query": query_text,
        "hits": len(hits),
        "comparisons": execution.comparisons,
        "merge_order": " ".join(execution.order),
    })
show_table(wildcard_query_rows)
assert kgram.wildcard("comput*")
assert wildcard_query_rows[0]["hits"] > 0

In [ ]:
spelling_rows = []
for misspelling in ["goverment", "brodband", "recieve", "managment", "sahres"]:
    nearest = kgram.nearest(misspelling, limit=3)
    spelling_rows.append({
        "misspelling": misspelling,
        "correction": kgram.correct(misspelling),
        "nearest_candidates": ", ".join(f"{term} (d={distance})" for term, distance in nearest),
    })
show_table(spelling_rows)
assert tolerant.levenshtein("goverment", "government") == 1
assert kgram.correct("goverment") == "government"

## 7. Evaluation

The 33 test queries cover simple, Boolean, negation, and morphological cases; 20 deliberately misspelled variants form the tolerant-retrieval set. Relevance judgments are deterministic whole-word predicates evaluated against **raw article text**, not against the index being scored.

For retrieved set $R$ and relevant set $G$:

$$P=\frac{|R\cap G|}{|R|},\qquad R_{ec}=\frac{|R\cap G|}{|G|},\qquad F_1=\frac{2PR_{ec}}{P+R_{ec}}$$

Macro averages weight each query equally. Micro averages pool document decisions first.

In [ ]:
qrels = queryset.build_qrels(docs)
query_inventory = defaultdict(int)
for query in queryset.QUERIES:
    query_inventory[query.kind] += 1

show_table([
    {"query_kind": kind, "count": count}
    for kind, count in sorted(query_inventory.items())
])
print(f"Test queries          : {len(queryset.QUERIES)}")
print(f"Misspelled variants   : {len(queryset.MISSPELLED)}")
print(f"Relevance judgments  : {sum(len(relevant) for relevant in qrels.values()):,}")

assert len(queryset.QUERIES) >= 30
assert len(queryset.MISSPELLED) >= 20
assert set(qrels) == {query.qid for query in queryset.QUERIES}
assert all(qrels.values()), "Every base query must have at least one relevant document."

In [ ]:
scores_by_index = {}
evaluation_summary = {}
evaluation_rows = []

for index_name, index in indexes.items():
    normalize = boolean.normalizer_for(preprocess.BY_NAME[index_name])
    scores = []
    for query in queryset.QUERIES:
        retrieved, _ = boolean.execute(query.text, index, normalize, optimize=True)
        query_score = evaluate.score(query.qid, set(retrieved), qrels[query.qid])
        scores.append(query_score)
        evaluation_rows.append({"index": index_name, "kind": query.kind, **query_score.as_row()})
    scores_by_index[index_name] = scores
    evaluation_summary[index_name] = evaluate.summarise(scores)

summary_rows = []
for index_name, summary in evaluation_summary.items():
    summary_rows.append({
        "index": index_name,
        "macro_precision": summary["macro"]["precision"],
        "macro_recall": summary["macro"]["recall"],
        "macro_f1": summary["macro"]["f1"],
        "micro_precision": summary["micro"]["precision"],
        "micro_recall": summary["micro"]["recall"],
        "micro_f1": summary["micro"]["f1"],
    })
show_table(summary_rows)

assert len(evaluation_rows) == len(indexes) * len(queryset.QUERIES)
assert evaluation_summary["stemmed"]["macro"]["recall"] > evaluation_summary["nostop"]["macro"]["recall"]

**Result interpretation.** Raw retrieval loses recall because case variants remain separate. Normalization and stop-word removal have identical effectiveness here because no test query contains a stop word. Stemming achieves the highest recall and F1 by conflating regular morphological variants, while lemmatization retains more precision but misses more surface variants.

In [ ]:
scores_by_kind = defaultdict(list)
for index_name, scores in scores_by_index.items():
    for query, query_score in zip(queryset.QUERIES, scores):
        scores_by_kind[(index_name, query.kind)].append(query_score)

kind_rows = [
    {
        "index": index_name,
        "kind": kind,
        "queries": len(scores),
        **evaluate.macro_average(scores),
    }
    for (index_name, kind), scores in sorted(scores_by_kind.items())
]
show_table(kind_rows)

stemmed_morph_recall = next(
    row["recall"] for row in kind_rows
    if row["index"] == "stemmed" and row["kind"] == "morphological"
)
nostop_morph_recall = next(
    row["recall"] for row in kind_rows
    if row["index"] == "nostop" and row["kind"] == "morphological"
)
assert stemmed_morph_recall > nostop_morph_recall

### 7.1 Exact versus tolerant retrieval on 20 misspelled queries

Each misspelled query is first executed unchanged against the stemmed index. Unknown **surface forms** are then corrected with the 3-gram/edit-distance index, after which the normal stemmed-query pipeline executes the repaired query. This ordering preserves the implemented behavior and avoids correcting against unreadable stems.

In [ ]:
tolerant_rows = []
exact_scores = []
repaired_scores = []

for misspelled in queryset.MISSPELLED:
    relevant = qrels[misspelled.base_qid]
    exact_docs, _ = boolean.execute(misspelled.text, stemmed_index, stemmed_normalize)

    corrections = {}
    for raw_term in sorted(set(boolean.all_terms(boolean.parse(misspelled.text)))):
        surface_term = raw_term.lower()
        if surface_term not in surface_index:
            corrections[raw_term] = kgram.correct(surface_term)

    repaired_query = misspelled.text
    for original, replacement in corrections.items():
        repaired_query = repaired_query.replace(original, replacement)
    repaired_docs, _ = boolean.execute(repaired_query, stemmed_index, stemmed_normalize)

    exact_score = evaluate.score(misspelled.qid, set(exact_docs), relevant)
    repaired_score = evaluate.score(misspelled.qid, set(repaired_docs), relevant)
    exact_scores.append(exact_score)
    repaired_scores.append(repaired_score)
    tolerant_rows.append({
        "qid": misspelled.qid,
        "corruption": misspelled.corruption,
        "misspelled_query": misspelled.text,
        "repaired_query": repaired_query,
        "exact_results": exact_score.retrieved,
        "tolerant_results": repaired_score.retrieved,
        "exact_recall": round(exact_score.recall, 4),
        "tolerant_recall": round(repaired_score.recall, 4),
        "tolerant_precision": round(repaired_score.precision, 4),
    })

show_table(tolerant_rows)
tolerant_summary = {
    "exact": evaluate.summarise(exact_scores),
    "tolerant": evaluate.summarise(repaired_scores),
}
show_table([
    {"mode": mode, **summary["macro"]}
    for mode, summary in tolerant_summary.items()
])
assert len(tolerant_rows) == 20
assert tolerant_summary["tolerant"]["macro"]["recall"] > tolerant_summary["exact"]["macro"]["recall"]

## 8. Visualize the experimental results

The plots below are generated directly from live notebook values. They summarize vocabulary reduction, retrieval effectiveness, and the recall gain from spelling repair.

In [ ]:
names = [row["index"] for row in index_rows]
figure, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].bar(names, [row["vocabulary_size"] for row in index_rows], color="#36688D")
axes[0].set_title("Vocabulary by configuration")
axes[0].set_ylabel("Distinct terms")
axes[0].tick_params(axis="x", rotation=25)

positions = list(range(len(names)))
width = 0.25
for offset, metric, color in [
    (-width, "precision", "#36688D"),
    (0, "recall", "#F49F05"),
    (width, "f1", "#2E8B57"),
]:
    axes[1].bar(
        [position + offset for position in positions],
        [evaluation_summary[name]["macro"][metric] for name in names],
        width=width,
        label=metric,
        color=color,
    )
axes[1].set_title("Macro effectiveness")
axes[1].set_ylim(0, 1.05)
axes[1].set_xticks(positions, names, rotation=25)
axes[1].legend()

modes = ["exact", "tolerant"]
axes[2].bar(
    modes,
    [tolerant_summary[mode]["macro"]["recall"] for mode in modes],
    color=["#A9A9A9", "#C44E52"],
)
axes[2].set_title("Misspelled-query recall")
axes[2].set_ylim(0, 1.05)
axes[2].set_ylabel("Macro recall")

figure.tight_layout()
figure_path = OUTPUT_DIR / "notebook_results.png"
figure.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure: {figure_path}")

## 9. Error analysis and limitations

- **Lexical judgments:** qrels are reproducible and independent of retrieval, but they are whole-word predicates rather than human semantic judgments. An article about an oil market that never says `oil` is counted non-relevant.
- **Stemming trade-off:** Porter improves morphological recall but can conflate unrelated senses such as `communism` and `community`; it also cannot resolve irregular morphology such as `children` and `child`.
- **Spelling false positives:** edit distance is form-based and may select a frequent but wrong neighbor. Rows such as `pirce -> piece` and `sahres -> sales` expose the precision/recall trade-off instead of hiding failures.
- **Tokenization:** punctuation is discarded and numbers with separators split (`45,000` becomes `45`, `000`).
- **Timing:** wall-clock measurements vary by machine; exact comparison counts are the primary deterministic optimization evidence.
- **Dataset licensing:** BBC article text is for non-commercial research use and is not intended for redistribution outside the submitted academic archive.

## 10. Generate and save final outputs

Notebook artifacts are written under `data/derived/notebook/` so they do not overwrite the canonical files generated by `experiments/run_experiments.py`. CSV uses UTF-8 and stable field order; JSON is indented and key-sorted for reproducible review.

In [ ]:
def write_csv_records(path: Path, rows: list[dict]) -> None:
    if not rows:
        raise ValueError(f"Cannot write empty result set to {path}")
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


notebook_summary = {
    "dataset": DATASET,
    "corpus": stats,
    "download": download_info,
    "indexes": index_rows,
    "evaluation": evaluation_summary,
    "evaluation_by_kind": kind_rows,
    "tolerant": tolerant_summary,
    "optimization_samples": optimization_rows,
    "wildcards": wildcard_rows,
}

summary_path = OUTPUT_DIR / "notebook_summary.json"
summary_path.write_text(json.dumps(notebook_summary, indent=2, sort_keys=True), encoding="utf-8")
write_csv_records(OUTPUT_DIR / "index_stats.csv", index_rows)
write_csv_records(OUTPUT_DIR / "evaluation.csv", evaluation_rows)
write_csv_records(OUTPUT_DIR / "tolerant.csv", tolerant_rows)
write_csv_records(OUTPUT_DIR / "boolean_optimization.csv", optimization_rows)

for output_path in sorted(OUTPUT_DIR.iterdir()):
    assert output_path.stat().st_size > 0, f"Empty output: {output_path}"
    print(f"{output_path.name:<28} {output_path.stat().st_size:>10,} bytes")

## 11. Run verification tests

This cell checks the notebook's main contracts and then runs the repository's complete pytest suite with the same Python interpreter as the notebook. A failure stops execution and identifies the broken invariant or test.

In [ ]:
for index_name, index in indexes.items():
    normalize = boolean.normalizer_for(preprocess.BY_NAME[index_name])
    index.check_invariants()
    for query in queryset.QUERIES:
        naive_docs, _ = boolean.execute(query.text, index, normalize, optimize=False)
        optimized_docs, _ = boolean.execute(query.text, index, normalize, optimize=True)
        assert set(naive_docs) == set(optimized_docs), (
            f"Strategy mismatch for {query.qid} in {index_name}"
        )

try:
    boolean.execute("comput* AND security", stemmed_index, stemmed_normalize)
    raise AssertionError("A wildcard without an expander must be rejected.")
except boolean.QuerySyntaxError:
    pass

empty_score = evaluate.score("empty", set(), {1})
assert (empty_score.precision, empty_score.recall, empty_score.f1) == (0.0, 0.0, 0.0)
assert summary_path.is_file() and summary_path.stat().st_size > 0

completed_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(completed_tests.stdout.strip())
print("Notebook invariants and repository tests passed.")

## 12. Processing summary

In [ ]:
source_size = ARCHIVE_PATH.stat().st_size if ARCHIVE_PATH.exists() else stats["raw_bytes"]
processing_summary = [
    {"item": "Source URL", "value": REMOTE_URL},
    {"item": "Input bytes", "value": f"{source_size:,}"},
    {"item": "Processed records", "value": f"{len(docs):,}"},
    {"item": "Base queries", "value": len(queryset.QUERIES)},
    {"item": "Misspelled queries", "value": len(queryset.MISSPELLED)},
    {"item": "Validation status", "value": "PASSED"},
    {"item": "Output location", "value": str(OUTPUT_DIR)},
]
show_table(processing_summary)
print("\nFinal-result preview:")
show_table(tolerant_rows, limit=5)

## 13. Conclusion

The notebook demonstrates the complete assignment pipeline over 2,225 BBC News articles while preserving the repository's tested implementation. Normalization reduces vocabulary duplication, stop-word removal primarily reduces index size, stemming gives the strongest morphological recall at a precision cost, postings-aware execution reduces avoidable comparisons (especially for negation), and 3-gram/edit-distance repair substantially improves recall on deliberately misspelled queries.

**Library boundary:** NLTK supplies English stop words, POS tagging, and WordNet lemmatization; Matplotlib supplies plots. Tokenization, the Porter stemmer, vocabulary, inverted index, postings merges, Boolean parser and optimizer, k-gram lookup, Levenshtein distance, relevance construction, and metric calculations are implemented in this repository.

For the BITS Virtual Lab demonstration, open this notebook, select the project `.venv` interpreter, choose **Run All**, and capture the final verification and processing-summary cells in the required screenshot.